In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2Model

model = GPT2Model.from_pretrained("gpt2")


In [2]:
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [3]:
print(model.config)

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.2",
  "use_cache": true,
  "vocab_size": 50257
}



In [4]:
from gpt import GPTConfig, GPTModel

gpt_cfg = GPTConfig.from_dict(model.config.to_dict() | {"qkv_bias": True})
print(gpt_cfg)

GPTConfig(vocab_size=50257, context_length=1024, embedding_dim=768, num_heads=12, num_layers=12, dropout=0.1, qkv_bias=True)


In [ ]:
gpt = GPTModel(gpt_cfg)
gpt.eval()

In [ ]:
model.state_dict().keys()

In [7]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return nn.Parameter(torch.tensor(right))

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

hf_sd = model.state_dict()
params = {
    "wte": hf_sd["wte.weight"].numpy(),
    "wpe": hf_sd["wpe.weight"].numpy(),
    "g": hf_sd["ln_f.weight"].numpy(),
    "b": hf_sd["ln_f.bias"].numpy(),
    "blocks": [],
}

for i in range(model.config.n_layer):
    block = {
        "attn": {
            "c_attn": {
                "w": hf_sd[f"h.{i}.attn.c_attn.weight"].numpy(),
                "b": hf_sd[f"h.{i}.attn.c_attn.bias"].numpy(),
            },
            "c_proj": {
                "w": hf_sd[f"h.{i}.attn.c_proj.weight"].numpy(),
                "b": hf_sd[f"h.{i}.attn.c_proj.bias"].numpy(),
            },
        },
        "mlp": {
            "c_fc": {
                "w": hf_sd[f"h.{i}.mlp.c_fc.weight"].numpy(),
                "b": hf_sd[f"h.{i}.mlp.c_fc.bias"].numpy(),
            },
            "c_proj": {
                "w": hf_sd[f"h.{i}.mlp.c_proj.weight"].numpy(),
                "b": hf_sd[f"h.{i}.mlp.c_proj.bias"].numpy(),
            },
        },
        "ln_1": {
            "g": hf_sd[f"h.{i}.ln_1.weight"].numpy(),
            "b": hf_sd[f"h.{i}.ln_1.bias"].numpy(),
        },
        "ln_2": {
            "g": hf_sd[f"h.{i}.ln_2.weight"].numpy(),
            "b": hf_sd[f"h.{i}.ln_2.bias"].numpy(),
        },
    }
    params["blocks"].append(block)

In [8]:

def load_weights_into_gpt(gpt, params):
    gpt.pos_embedding.weight = assign(gpt.pos_embedding.weight, params["wpe"])
    gpt.token_embedding.weight = assign(gpt.token_embedding.weight, params["wte"])

    for b in range(len(params["blocks"])):
        qkv_w = params["blocks"][b]["attn"]["c_attn"]["w"].T
        gpt.transformer_blocks[b].attention.qkv.weight = assign(
            gpt.transformer_blocks[b].attention.qkv.weight, qkv_w
        )

        qkv_b = params["blocks"][b]["attn"]["c_attn"]["b"]
        gpt.transformer_blocks[b].attention.qkv.bias = assign(
            gpt.transformer_blocks[b].attention.qkv.bias, qkv_b
        )

        gpt.transformer_blocks[b].attention.out_proj.weight = assign(
            gpt.transformer_blocks[b].attention.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T,
        )
        gpt.transformer_blocks[b].attention.out_proj.bias = assign(
            gpt.transformer_blocks[b].attention.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"],
        )

        gpt.transformer_blocks[b].ff.net[0].weight = assign(
            gpt.transformer_blocks[b].ff.net[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T,
        )
        gpt.transformer_blocks[b].ff.net[0].bias = assign(
            gpt.transformer_blocks[b].ff.net[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"],
        )
        gpt.transformer_blocks[b].ff.net[2].weight = assign(
            gpt.transformer_blocks[b].ff.net[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T,
        )
        gpt.transformer_blocks[b].ff.net[2].bias = assign(
            gpt.transformer_blocks[b].ff.net[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"],
        )

        gpt.transformer_blocks[b].ln1.scale = assign(
            gpt.transformer_blocks[b].ln1.scale, params["blocks"][b]["ln_1"]["g"]
        )
        gpt.transformer_blocks[b].ln1.shift = assign(
            gpt.transformer_blocks[b].ln1.shift, params["blocks"][b]["ln_1"]["b"]
        )
        gpt.transformer_blocks[b].ln2.scale = assign(
            gpt.transformer_blocks[b].ln2.scale, params["blocks"][b]["ln_2"]["g"]
        )
        gpt.transformer_blocks[b].ln2.shift = assign(
            gpt.transformer_blocks[b].ln2.shift, params["blocks"][b]["ln_2"]["b"]
        )

    gpt.final_layer_norm.scale = assign(gpt.final_layer_norm.scale, params["g"])
    gpt.final_layer_norm.shift = assign(gpt.final_layer_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])


In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [12]:
from gpt import generate
import tiktoken
from utils import text_to_tokens, tokens_to_text

tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(43)
token_ids = generate(
    model=gpt,
    input_ids=text_to_tokens("Every effort moves you", tokenizer).to(device),
    max_length=50,
    context_length=gpt_cfg.context_length,
    top_k=50,
    temperature=1.0,
)

print(tokens_to_text(token_ids, tokenizer))

Every effort moves you toward the end of the journey and you can always turn back to any of the other items and have a new beginning of your journey. Your skill sets can change with each day and different skill levels, but they are all connected in a way that is


In [23]:
with open("alice.txt", "r") as f:
    text = f.read()

print("Number of characters:", len(text))
print("Number of tokens:", len(tokenizer.encode(text)))

Number of characters: 143009
Number of tokens: 41769


In [28]:
from utils import create_dataloader_v1

train_ratio = 0.8
train_size = int(len(text) * train_ratio)
train_data = text[:train_size]
valid_data = text[train_size:]

torch.manual_seed(42)
train_dataloader = create_dataloader_v1(
    train_data,
    context_size=gpt_cfg.context_length,
    stride=gpt_cfg.context_length,
    batch_size=2,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
valid_dataloader = create_dataloader_v1(
    valid_data,
    context_size=gpt_cfg.context_length,
    stride=gpt_cfg.context_length,
    batch_size=2,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

In [ ]:
from train import evaluate_model

torch.manual_seed(42)
train_loss, valid_loss = evaluate_model(gpt, train_dataloader, valid_dataloader, None)
print(f"Train Loss: {train_loss:.4f}")
print(f"Valid Loss: {valid_loss:.4f}")

Train Loss: 2.8196
Valid Loss: 2.8342
